### ***Silver Processing with SCD Type 2 Approach Applied***

In [ ]:
from functools import reduce
from pyspark.sql.functions import *

catalog_db = "FABRIC_COE_WORKLOAD.stock_market_analysis_LH"

bronze_tables = [
    "brz_aapl_stock",
    "brz_amd_stock",
    "brz_amzn_stock",
    "brz_crm_stock",
    "brz_intc_stock",
    "brz_jpm_stock",
    "brz_meta_stock",
    "brz_msft_stock",
    "brz_nflx_stock",
    "brz_nvda_stock",
    "brz_orcl_stock",
    "brz_pltr_stock",
    "brz_pypl_stock",
    "brz_tsla_stock",
    "brz_wmt_stock"
]

dfs = [spark.table(f"{catalog_db}.bronze.{table}")for table in bronze_tables]

# Append all Bronze stock tables
df_silver = functools.reduce(lambda df1, df2: df1.unionByName(df2),dfs)

# Basic data quality checks
df_silver = (
    df_silver
    .dropDuplicates(["ticker", "timestamp", "effective_from"])
    .filter(col("ticker").isNotNull())
    .filter(col("timestamp").isNotNull())
    .filter(col("close").isNotNull())
    .filter(col("close") > 0)
)


#Select current Bronze records
df_source = (
    df_silver
    .filter(col("is_current") == True)
    .drop("is_current", "effective_from", "effective_to")
    .withColumn("is_current", lit(True))
    .withColumn("effective_from", current_timestamp())
    .withColumn("effective_to", lit(None).cast("timestamp"))
)

#Create a temporary source view
df_source.createOrReplaceTempView("vw_silver_source")

#Silver Table name 
table_name = f"{catalog_db}.silver.slv_stock_prices"

#Check whether the Silver table exists
if not spark.catalog.tableExists(table_name):

    # Initial load
    (
        df_source.write
        .format("delta")
        .mode("errorifexists")
        .option("mergeSchema", True)
        .saveAsTable(table_name)
    )

    print("Silver stock Delta table created successfully")

else:
    print("Silver table exists. Applying SCD Type 2 logic.")

    # 1. Expire current records when stock values change
    spark.sql(f"""
        MERGE INTO {table_name} AS target
        USING vw_silver_source AS source

        ON  target.ticker = source.ticker
        AND target.timestamp = source.timestamp
        AND target.is_current = true

        WHEN MATCHED AND (
               NOT (target.currency <=> source.currency)
            OR NOT (target.trade_datetime <=> source.trade_datetime)
            OR NOT (target.open <=> source.open)
            OR NOT (target.high <=> source.high)
            OR NOT (target.low <=> source.low)
            OR NOT (target.close <=> source.close)
            OR NOT (target.volume <=> source.volume)
            OR NOT (target.adj_close <=> source.adj_close)
        )

        THEN UPDATE SET
            target.is_current = false,
            target.effective_to = current_timestamp()
    """)

    # 2. Identify new keys and changed records
    spark.sql(f"""
        CREATE OR REPLACE TEMP VIEW vw_new_silver_versions AS

        SELECT source.*
        FROM vw_silver_source AS source

        LEFT ANTI JOIN (
            SELECT ticker, timestamp
            FROM {table_name}
            WHERE is_current = true
        ) AS target

        ON  source.ticker = target.ticker
        AND source.timestamp = target.timestamp
    """)

    # 3. Insert new records and new current versions
    spark.sql(f"""
        INSERT INTO {table_name} (
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            is_current, effective_from, effective_to
        )

        SELECT
            ticker, currency, timestamp, trade_datetime,
            open, high, low, close, volume, adj_close,
            ingested_at, file_path,
            true, current_timestamp(), CAST(NULL AS TIMESTAMP)

        FROM vw_new_silver_versions
    """)

    print("Silver stock SCD Type 2 processing completed")

#Display all Silver versions
display(
    spark.table(table_name)
    .orderBy("ticker", "timestamp", "effective_from")
)

### ***Check Distinct Stock Ticker values for Verification***

In [ ]:
%%sql
select distinct ticker from stock_market_analysis_LH.silver.slv_stock_prices;

StatementMeta(, 3e9b59b6-ace9-43be-ac6a-430405e11b5f, 9, Finished, Available, Finished, False)

<Spark SQL result set with 15 rows and 1 fields>